# CallGuard AI - Notebook 01: Data Understanding

### Objective
Understand the structure, size, vocabulary, and characteristics of all datasets utilized across the CallGuard AI platform:
- **CLINC150 (clinc_oos)**: 150 intent classes across 10 domains (banking, credit cards, travel, etc.) plus out-of-scope queries.
- **BANKING77**: 77 fine-grained banking domain customer service intent classes.
- **FTC Robocall Complaints**: Federal Trade Commission telemarketing and illegal robocall reports.
- **ASVspoof**: Audio spoofing, replay attack, and synthetic voice detection benchmark (telephony voice security).
- **CallGuard Custom Dataset**: Multi-turn synthesized and annotated telephony conversations spanning recruitment, job scams, OTP theft, and legitimate calls.

In [ ]:
# Cell 2: Install dependencies
!pip install -q datasets pandas matplotlib seaborn scikit-learn

In [ ]:
# Cell 3: Import libraries
import os
import json
import logging
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)

print("Environment libraries initialized.")

In [ ]:
# Cell 4: Load CLINC150 from HuggingFace
from datasets import load_dataset

try:
    print("Fetching CLINC150 (clinc_oos, plus) from Hugging Face...")
    clinc_ds = load_dataset("clinc_oos", "plus")
    clinc_train_df = clinc_ds["train"].to_pandas()
    clinc_val_df = clinc_ds["validation"].to_pandas()
    clinc_test_df = clinc_ds["test"].to_pandas()
    clinc_df = pd.concat([clinc_train_df, clinc_val_df, clinc_test_df], ignore_index=True)
    print(f"Loaded CLINC150 successfully: {len(clinc_df)} total records across train/val/test.")
except Exception as e:
    print(f"Notice: HuggingFace fetch skipped ({e}). Initializing benchmark fallback samples...")
    clinc_df = pd.DataFrame([
        {"text": "what is my balance in checking account", "intent": "balance", "split": "train"},
        {"text": "transfer 200 dollars to savings", "intent": "transfer", "split": "train"},
        {"text": "lost my credit card need to freeze it", "intent": "freeze_account", "split": "train"},
        {"text": "report fraudulent transaction on statement", "intent": "report_fraud", "split": "validation"},
        {"text": "what is the routing number for international wire", "intent": "routing", "split": "test"},
    ])

In [ ]:
# Cell 5: CLINC150 exploration
print("=== CLINC150 Dataset Exploration ===")
print("Shape:", clinc_df.shape)
print("Columns:", clinc_df.columns.tolist())
if "intent" in clinc_df.columns:
    unique_intents = clinc_df["intent"].nunique()
    print(f"Total Unique Intent Classes: {unique_intents}")

print("\nSample Rows:")
display(clinc_df.head(5))

# Calculate utterance length statistics
clinc_df["char_len"] = clinc_df["text"].str.len()
clinc_df["word_len"] = clinc_df["text"].str.split().str.len()
print("\nUtterance Word Count Stats:")
print(clinc_df["word_len"].describe())

In [ ]:
# Cell 6: Load BANKING77
try:
    print("Fetching BANKING77 from Hugging Face...")
    banking_ds = load_dataset("PolyAI/banking77")
    banking_train = banking_ds["train"].to_pandas()
    banking_test = banking_ds["test"].to_pandas()
    banking_df = pd.concat([banking_train, banking_test], ignore_index=True)
    
    # Map label IDs to category names if available
    if hasattr(banking_ds["train"].features["label"], "int2str"):
        banking_df["label_name"] = banking_df["label"].apply(banking_ds["train"].features["label"].int2str)
    print(f"Loaded BANKING77 successfully: {len(banking_df)} total records.")
except Exception as e:
    print(f"Notice: HuggingFace fetch skipped ({e}). Initializing fallback samples...")
    banking_df = pd.DataFrame([
        {"text": "card payment pending for 3 days", "label_name": "pending_card_payment", "split": "train"},
        {"text": "where is my replacement debit card", "label_name": "card_arrival", "split": "train"},
        {"text": "activate my virtual credit card", "label_name": "activate_my_card", "split": "train"},
        {"text": "unauthorized charge from unknown store", "label_name": "card_payment_not_recognised", "split": "test"},
    ])

In [ ]:
# Cell 7: BANKING77 exploration
print("=== BANKING77 Dataset Exploration ===")
print("Shape:", banking_df.shape)
label_col = "label_name" if "label_name" in banking_df.columns else "label"
print(f"Unique Banking Intents: {banking_df[label_col].nunique()}")

banking_df["word_len"] = banking_df["text"].str.split().str.len()
print("\nBanking77 Word Length Summary:")
print(banking_df["word_len"].describe())

print("\nSample Banking77 Records:")
display(banking_df[["text", label_col]].head(5))

In [ ]:
# Cell 8: Load/inspect FTC data (Placeholder with manual download instructions)
"""
FTC Robocall Complaint Dataset Instructions:
----------------------------------------------------
1. Visit the Federal Trade Commission's Do Not Call (DNC) Reported Data:
   https://www.ftc.gov/enforcement/data-visualizations/national-do-not-call-registry-data-book
2. Download monthly or annual robocall complaint CSV dumps.
3. Save the raw file to `ml/datasets/ftc/ftc_robocall_complaints.csv`.
4. Columns typically include:
   - Company_Phone_Number
   - Consumer_City / Consumer_State
   - Violation_Date / Time
   - Subject_Matter (e.g. Debt reduction, Medical, Warranties, Impersonation)
   - Recorded_Message_Flag (Y/N)
"""

ftc_sample_path = Path("ml/datasets/ftc/ftc_robocall_complaints.csv")
if ftc_sample_path.exists():
    ftc_df = pd.read_csv(ftc_sample_path)
    print("Loaded local FTC robocalls dataset:", ftc_df.shape)
    display(ftc_df.head(3))
else:
    print("FTC dataset placeholder: Mocking FTC Telemarketing Complaint structure for pipeline testing...")
    ftc_df = pd.DataFrame([
        {"caller_phone": "+18005550199", "subject": "Reducing credit card interest rate", "robocall_flag": "Y", "recorded_message": True},
        {"caller_phone": "+18885550142", "subject": "Extended vehicle auto warranty", "robocall_flag": "Y", "recorded_message": True},
        {"caller_phone": "+12025550187", "subject": "IRS tax debt settlement notice", "robocall_flag": "Y", "recorded_message": False},
        {"caller_phone": "+13125550111", "subject": "Home solar panel rebate program", "robocall_flag": "Y", "recorded_message": True},
    ])
    display(ftc_df)

In [ ]:
# Cell 9: Custom CallGuard dataset placeholder
callguard_path = Path("ml/datasets/callguard/synthetic_conversations.jsonl")

if callguard_path.exists():
    with open(callguard_path, "r", encoding="utf-8") as f:
        cg_records = [json.loads(line) for line in f if line.strip()]
    callguard_df = pd.DataFrame(cg_records)
    print(f"Loaded CallGuard Custom Dataset from {callguard_path}: {len(callguard_df)} calls.")
else:
    print("Generating bootstrap CallGuard dataset using generator script...")
    from ml.scripts.synthetic_data_generator import generate_dataset
    cg_records = generate_dataset(output_path=str(callguard_path), count_per_scenario=10)
    callguard_df = pd.DataFrame(cg_records)

print("\nCallGuard Dataset Structure:")
print("Columns:", callguard_df.columns.tolist())
display(callguard_df[["id", "scenario", "caller_type", "intent", "risk_level", "action"]].head(5))

In [ ]:
# Cell 10: Dataset statistics summary table
summary_data = [
    {
        "Dataset": "CLINC150 (Plus)",
        "Domain": "Cross-Domain Task Intents & OOS",
        "Total Records": len(clinc_df),
        "Unique Classes": clinc_df["intent"].nunique() if "intent" in clinc_df.columns else 0,
        "Mean Word Count": round(clinc_df["word_len"].mean(), 2) if "word_len" in clinc_df.columns else 0,
        "Primary Modality": "Single-turn text queries"
    },
    {
        "Dataset": "BANKING77",
        "Domain": "Banking & Financial Services",
        "Total Records": len(banking_df),
        "Unique Classes": banking_df[label_col].nunique() if label_col in banking_df.columns else 0,
        "Mean Word Count": round(banking_df["word_len"].mean(), 2) if "word_len" in banking_df.columns else 0,
        "Primary Modality": "Customer banking text queries"
    },
    {
        "Dataset": "FTC Robocall Registry",
        "Domain": "Telemarketing & Phone Fraud Reports",
        "Total Records": len(ftc_df),
        "Unique Classes": ftc_df["subject"].nunique() if "subject" in ftc_df.columns else 0,
        "Mean Word Count": 0,
        "Primary Modality": "Metadata & Phone numbers"
    },
    {
        "Dataset": "CallGuard Custom",
        "Domain": "Telephony Security, Recruitment & Scams",
        "Total Records": len(callguard_df),
        "Unique Classes": callguard_df["intent"].nunique() if "intent" in callguard_df.columns else 0,
        "Mean Word Count": round(callguard_df["full_transcript"].str.split().str.len().mean(), 2) if "full_transcript" in callguard_df.columns else 0,
        "Primary Modality": "Multi-turn dialog transcripts"
    }
]

stats_summary_df = pd.DataFrame(summary_data)
display(stats_summary_df)

In [ ]:
# Cell 11: Class distribution plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot CallGuard Scenario distribution
if "scenario" in callguard_df.columns:
    scenario_counts = callguard_df["scenario"].value_counts()
    sns.barplot(x=scenario_counts.values, y=scenario_counts.index, ax=axes[0], palette="crest")
    axes[0].set_title("CallGuard Operational Scenarios Distribution", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Sample Count")

# Plot CallGuard Risk Level distribution
if "risk_level" in callguard_df.columns:
    risk_counts = callguard_df["risk_level"].value_counts()
    sns.barplot(x=risk_counts.index, y=risk_counts.values, ax=axes[1], palette="mako")
    axes[1].set_title("CallGuard Risk Level Distribution", fontsize=12, fontweight="bold")
    axes[1].set_ylabel("Call Count")

plt.tight_layout()
plt.show()

# Cell 12: Conclusions

### Key Takeaways from Data Understanding:
1. **Domain Diversity**: Public benchmarks like CLINC150 and BANKING77 provide exceptional coverage of banking and general intents, but focus on short text queries.
2. **Telephony Dialog Reality**: Real telephony security requires conversational transcripts featuring multi-turn greetings, interruptions, and social engineering patterns.
3. **CallGuard Synthetic Dataset Role**: Spanning 11 operational scenarios (AI recruiters, human scammers, OTP theft, legitimate deliveries), our custom dataset bridges the gap between single queries and complete phone call transcripts.
4. **Next Steps**: Clean, normalize, and tokenize texts in `02_data_cleaning.ipynb` prior to feature extraction and model training.